In [659]:
import sqlite3
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import torch.nn as nn

In [ ]:
db_path = Path(
    "/Users/linda/Documents/MLDS/Winter 2026/Data Mining/Group Project/database.sqlite"
)
conn = sqlite3.connect(db_path.as_posix())
df_agg = pd.read_sql_query("SELECT * FROM Match_Aggregated", conn)
df_match = pd.read_sql_query("SELECT * FROM Match_Cleaned", conn)
conn.close()

In [661]:
df_match_fc_bayern_munich = df_agg[
    (df_agg["match_api_id"] == 2002277) & (df_agg["team_side"] == "away")
]
df_match_fc_bayern_munich.head()

,match_api_id,date,position_col,player_api_id,team_side,position_no,id,player_fifa_api_id,overall_rating,potential,...,away_team_chanceCreationShooting,away_team_chanceCreationPositioningClass,away_team_defencePressure,away_team_defenceAggression,away_team_defenceTeamWidth,away_team_defenceDefenderLineClass,away_team_team_long_name,away_team_team_short_name,player_x,player_y
406790,2002277,2016-02-14 00:00:00,away_player_7,30834,away,7,16461,9014,89.0,89.0,...,22.0,1.0,72.0,53.0,59.0,0.0,FC Bayern Munich,BMU,2.0,8.0
406794,2002277,2016-02-14 00:00:00,away_player_2,30894,away,2,143229,121939,87.0,87.0,...,22.0,1.0,72.0,53.0,59.0,0.0,FC Bayern Munich,BMU,2.0,3.0
406890,2002277,2016-02-14 00:00:00,away_player_6,49939,away,6,17411,181872,85.0,85.0,...,22.0,1.0,72.0,53.0,59.0,0.0,FC Bayern Munich,BMU,5.0,6.0
406927,2002277,2016-02-14 00:00:00,away_player_11,93447,away,11,150542,188545,88.0,89.0,...,22.0,1.0,72.0,53.0,59.0,0.0,FC Bayern Munich,BMU,5.0,11.0
406961,2002277,2016-02-14 00:00:00,away_player_8,116772,away,8,170800,189596,86.0,88.0,...,22.0,1.0,72.0,53.0,59.0,0.0,FC Bayern Munich,BMU,4.0,8.0


In [662]:
df_match_real_madrid = df_agg[
    (df_agg["match_api_id"] == 2030529) & (df_agg["team_side"] == "away")
]
df_match_real_madrid.head()

,match_api_id,date,position_col,player_api_id,team_side,position_no,id,player_fifa_api_id,overall_rating,potential,...,away_team_chanceCreationShooting,away_team_chanceCreationPositioningClass,away_team_defencePressure,away_team_defenceAggression,away_team_defenceTeamWidth,away_team_defenceDefenderLineClass,away_team_team_long_name,away_team_team_short_name,player_x,player_y
425882,2030529,2016-05-14 00:00:00,away_player_3,25921,away,3,142272,120533,84.0,84.0,...,63.0,1.0,52.0,60.0,63.0,0.0,Real Madrid CF,REA,4.0,3.0
425887,2030529,2016-05-14 00:00:00,away_player_10,26166,away,10,93534,165153,86.0,86.0,...,63.0,1.0,52.0,60.0,63.0,0.0,Real Madrid CF,REA,5.0,10.0
425907,2030529,2016-05-14 00:00:00,away_player_5,28467,away,5,110682,176676,83.0,83.0,...,63.0,1.0,52.0,60.0,63.0,0.0,Real Madrid CF,REA,8.0,3.0
425916,2030529,2016-05-14 00:00:00,away_player_11,30893,away,11,33331,20801,93.0,93.0,...,63.0,1.0,52.0,60.0,63.0,0.0,Real Madrid CF,REA,7.0,10.0
425925,2030529,2016-05-14 00:00:00,away_player_4,30962,away,4,161328,155862,87.0,87.0,...,63.0,1.0,52.0,60.0,63.0,0.0,Real Madrid CF,REA,6.0,3.0


# Model 1

In [663]:
class FormationTower(nn.Module):
    def __init__(self):
        super().__init__()
        # Phi: (B, 9, 2) -> (B, 9, 64)
        self.phi = nn.Sequential(
            nn.Linear(in_features=2, out_features=64),
            nn.LeakyReLU(),
            nn.Linear(in_features=64, out_features=64),
            nn.LeakyReLU(),
            nn.Linear(in_features=64, out_features=64),
            nn.LeakyReLU(),
        )

        # Rho: (B, 64 + 2) -> (B, 64)
        # We match in_features to the phi output (64) + position (2)
        self.rho = nn.Sequential(
            nn.Linear(in_features=64 + 2, out_features=128),
            nn.BatchNorm1d(128),  # Must match out_features above
            nn.LeakyReLU(),
            nn.Linear(in_features=128, out_features=128),
            nn.BatchNorm1d(128),
            nn.LeakyReLU(),
            nn.Linear(in_features=128, out_features=64),  # Final embedding dim: 64
        )

    def forward(self, formation, position):
        x = self.phi(formation)  # (Batch, 9, 64)
        x = torch.sum(x, dim=1)  # (Batch, 64)

        x = torch.concat([x, position], dim=1)  # (Batch, 66)
        return self.rho(x)


class PlayerTower(nn.Module):
    def __init__(self):
        super().__init__()
        self.mlp = nn.Sequential(
            nn.Linear(in_features=31, out_features=128),
            nn.BatchNorm1d(128),
            nn.LeakyReLU(),
            nn.Linear(in_features=128, out_features=128),
            nn.BatchNorm1d(128),
            nn.LeakyReLU(),
            nn.Linear(in_features=128, out_features=64),  # Final embedding dim: 64
        )

    def forward(self, player):
        return self.mlp(player)

In [664]:
class TwoTower(nn.Module):
    def __init__(self):
        super().__init__()
        self.formation_tower = FormationTower()
        self.player_tower = PlayerTower()

    def forward(self, formation, position, player):
        formation_embedding = self.formation_tower(formation, position)
        player_embedding = self.player_tower(player)

        f_norm = formation_embedding / (
            torch.linalg.norm(formation_embedding, dim=1, keepdim=True) + 1e-8
        )
        p_norm = player_embedding / (
            torch.linalg.norm(player_embedding, dim=1, keepdim=True) + 1e-8
        )

        similarity_score = (f_norm * p_norm).sum(dim=1)

        return f_norm, p_norm, similarity_score

In [665]:
player_stats_cols = [
    "preferred_foot",
    "attacking_work_rate",
    "defensive_work_rate",
    "crossing",
    "finishing",
    "heading_accuracy",
    "short_passing",
    "volleys",
    "dribbling",
    "curve",
    "free_kick_accuracy",
    "long_passing",
    "ball_control",
    "acceleration",
    "sprint_speed",
    "agility",
    "reactions",
    "balance",
    "shot_power",
    "jumping",
    "stamina",
    "strength",
    "long_shots",
    "aggression",
    "interceptions",
    "positioning",
    "vision",
    "penalties",
    "marking",
    "standing_tackle",
    "sliding_tackle",
]

In [666]:
import plotly.graph_objects as go
import numpy as np
import torch
import pandas as pd


def visualize_tactical_fit(custom_coords, squad_df, model, device, player_stats_cols):
    """
    custom_coords: List of tuples [(x1, y1), (x2, y2), ... (x10, y10)]
    squad_df: DataFrame containing the players you want to test (e.g., FC Bayern)
    """
    model.eval()

    # 1. Prepare Coordinates
    coords_np = np.array(custom_coords, dtype=float)
    # Normalize globally (consistent with training: x centered at 5, y at 7)
    coords_norm = coords_np.copy()
    coords_norm[:, 0] = (coords_norm[:, 0] - 5.0) / 4.0
    coords_norm[:, 1] = (coords_norm[:, 1] - 7.0) / 4.0

    all_rankings = []

    # 2. Process each of the 10 positions
    for i in range(len(coords_norm)):
        target_pos_val = coords_norm[i]
        teammates_pos_vals = np.delete(coords_norm, i, axis=0)

        # Calculate Relative Geometry (matching advanced model logic)
        relative_vectors = teammates_pos_vals - target_pos_val
        distances = np.linalg.norm(relative_vectors, axis=1) / 2.5
        angles = np.arctan2(relative_vectors[:, 1], relative_vectors[:, 0]) / np.pi

        formation_input = np.stack([distances, angles], axis=1)[:9]

        # Convert to Tensors
        target_pos_tensor = torch.tensor([target_pos_val], dtype=torch.float32).to(
            device
        )
        formation_tensor = torch.tensor([formation_input], dtype=torch.float32).to(
            device
        )

        with torch.no_grad():
            # Encode the "Job Requirement"
            f_embed = model.formation_tower(formation_tensor, target_pos_tensor)
            f_norm = f_embed / (torch.linalg.norm(f_embed, dim=1, keepdim=True) + 1e-8)

            # Score all players in the squad for THIS specific position
            scores = []
            for _, player_row in squad_df.iterrows():
                p_stats = torch.tensor(
                    [player_row[player_stats_cols].values], dtype=torch.float32
                ).to(device)
                p_embed = model.player_tower(p_stats)
                p_norm = p_embed / (
                    torch.linalg.norm(p_embed, dim=1, keepdim=True) + 1e-8
                )

                score = torch.matmul(f_norm, p_norm.T).item()
                scores.append((player_row["player_name"], score))

        # Sort and take top 5 for the hover tooltip
        scores.sort(key=lambda x: x[1], reverse=True)
        top_list = "<br>".join([f"{name}: {s:.3f}" for name, s in scores[:5]])
        all_rankings.append(top_list)

    # 3. Create Plotly Visualization
    fig = go.Figure()

    fig.add_trace(
        go.Scatter(
            x=coords_np[:, 0],
            y=coords_np[:, 1],
            mode="markers+text",
            marker=dict(
                size=25, color="RoyalBlue", line=dict(width=2, color="DarkSlateGrey")
            ),
            text=[f"Pos {i + 1}" for i in range(10)],
            textposition="top center",
            hoverinfo="text",
            hovertext=[f"<b>Top Candidates:</b><br>{rank}" for rank in all_rankings],
        )
    )

    # Clean up the layout to look like a pitch
    fig.update_layout(
        title="Tactical Fit Analysis: Hover to see ideal players",
        xaxis=dict(title="Pitch Width (X)", range=[0, 10], showgrid=False),
        yaxis=dict(title="Pitch Length (Y)", range=[0, 14], showgrid=False),
        width=700,
        height=900,
        template="plotly_white",
    )

    fig.show()


# --- EXAMPLE USAGE ---
# Define a 4-2-3-1 (approximate coords)
modern_4231 = [
    (2, 3),
    (8, 3),
    (4, 3),
    (6, 3),  # Back 4
    (4, 6),
    (6, 6),  # Midfield 2
    (2, 9),
    (5, 9),
    (8, 9),  # Attacking Mid 3
    (5, 12),  # Striker
]

classic_442 = [
    # Back 4 (Defense)
    (2, 3),  # Left Back (LB)
    (4, 3),  # Left Center Back (LCB)
    (6, 3),  # Right Center Back (RCB)
    (8, 3),  # Right Back (RB)
    # Midfield 4
    (2, 7),  # Left Midfield (LM / LW)
    (4, 7),  # Left Central Midfield (LCM)
    (6, 7),  # Right Central Midfield (RCM)
    (8, 7),  # Right Midfield (RM / RW)
    # Front 2 (Strikers)
    (4, 11),  # Left Striker (LS)
    (6, 11),  # Right Striker (RS)
]

attacking_433 = [
    # Back 4
    (2, 3),  # Left Back (LB)
    (4, 3),  # Left Center Back (LCB)
    (6, 3),  # Right Center Back (RCB)
    (8, 3),  # Right Back (RB)
    # Midfield 3
    (3, 7),  # Defensive Midfielder (CDM) - The "Pivot"
    (5, 7),  # Left Central Midfielder (LCM)
    (7, 7),  # Right Central Midfielder (RCM)
    # Front 3
    (2, 11),  # Left Winger (LW)
    (8, 11),  # Right Winger (RW)
    (5, 11),  # Center Forward (ST)
]

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Load the Simple Model
model = TwoTower().to(device)
model_path = "best_soccer_model.pth"
state_dict = torch.load(model_path, map_location=device, weights_only=True)
model.load_state_dict(state_dict)

visualize_tactical_fit(
    modern_4231, df_match_fc_bayern_munich, model, device, player_stats_cols
)
# visualize_tactical_fit(
#     attacking_433, df_match_fc_bayern_munich, model, device, player_stats_cols
# )

# Model 2

In [667]:
class PositionTower(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_features=2, out_features=64),
            nn.BatchNorm1d(64),
            nn.LeakyReLU(),
            nn.Linear(in_features=64, out_features=128),
            nn.BatchNorm1d(128),
            nn.LeakyReLU(),
            nn.Linear(in_features=128, out_features=64),
        )

    def forward(self, x):
        return self.net(x)


class PlayerTower(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.BatchNorm1d(31),
            nn.Linear(in_features=31, out_features=128),
            nn.BatchNorm1d(128),
            nn.LeakyReLU(),
            nn.Linear(in_features=128, out_features=128),
            nn.BatchNorm1d(128),
            nn.LeakyReLU(),
            nn.Linear(in_features=128, out_features=64),
        )

    def forward(self, x):
        return self.net(x)


class SimpleTwoTower(nn.Module):
    def __init__(self):
        super().__init__()
        self.position_tower = PositionTower()
        self.player_tower = PlayerTower()

    def forward(self, position, player):
        pos_emb = self.position_tower(position)
        plr_emb = self.player_tower(player)

        p_norm = pos_emb / (torch.linalg.norm(pos_emb, dim=1, keepdim=True) + 1e-8)
        u_norm = plr_emb / (torch.linalg.norm(plr_emb, dim=1, keepdim=True) + 1e-8)

        return p_norm, u_norm

In [668]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Load the Simple Model
model = SimpleTwoTower().to(device)
model_path = "simple_soccer_model.pth"  # Ensure you saved the simple model here
state_dict = torch.load(model_path, map_location=device, weights_only=True)
model.load_state_dict(state_dict)

<All keys matched successfully>

In [669]:
import plotly.graph_objects as go
import numpy as np
import torch
import pandas as pd


def visualize_simple_tactical_fit(
    custom_coords, squad_df, model, device, player_stats_cols
):
    """
    custom_coords: List of tuples [(x, y), ...] for any number of positions
    squad_df: DataFrame containing the squad to rank
    """
    model.eval()

    # 1. Prepare Coordinates
    coords_np = np.array(custom_coords, dtype=float)

    # Normalize globally exactly like the simple model training
    coords_norm = coords_np.copy()
    coords_norm[:, 0] = (coords_norm[:, 0] - 5.0) / 4.0
    coords_norm[:, 1] = (coords_norm[:, 1] - 7.0) / 4.0

    all_rankings = []

    # 2. Process each position independently (Simple Model Logic)
    for i in range(len(coords_norm)):
        # Convert single position to tensor (Batch size 1)
        target_pos_tensor = torch.tensor([coords_norm[i]], dtype=torch.float32).to(
            device
        )

        with torch.no_grad():
            # Encode the 'Position Requirement'
            p_emb = model.position_tower(target_pos_tensor)
            p_norm = p_emb / (torch.linalg.norm(p_emb, dim=1, keepdim=True) + 1e-8)

            # Score all players in the squad
            scores = []
            for _, player_row in squad_df.iterrows():
                # Extract player stats
                p_stats = torch.tensor(
                    [player_row[player_stats_cols].values], dtype=torch.float32
                ).to(device)

                # Encode Player (BatchNorm here uses moving averages due to model.eval())
                u_emb = model.player_tower(p_stats)
                u_norm = u_emb / (torch.linalg.norm(u_emb, dim=1, keepdim=True) + 1e-8)

                # Calculate Similarity
                score = torch.matmul(p_norm, u_norm.T).item()
                scores.append((player_row["player_name"], score))

        # Sort leaderboard for this specific dot
        scores.sort(key=lambda x: x[1], reverse=True)
        top_list = "<br>".join([f"{name}: {s:.4f}" for name, s in scores[:5]])
        all_rankings.append(top_list)

    # 3. Create Plotly Visualization
    fig = go.Figure()

    # Draw the pitch boundaries for context
    fig.add_shape(type="rect", x0=0, y0=0, x1=10, y1=14, line=dict(color="Black"))

    fig.add_trace(
        go.Scatter(
            x=coords_np[:, 0],
            y=coords_np[:, 1],
            mode="markers+text",
            marker=dict(
                size=30,
                color="FireBrick",  # Changed color to distinguish from advanced model
                line=dict(width=2, color="DarkSlateGrey"),
                opacity=0.8,
            ),
            text=[f"Pos {i + 1}" for i in range(len(custom_coords))],
            textposition="top center",
            hoverinfo="text",
            hovertext=[
                f"<b>Top Fits (Simple Model):</b><br>{rank}" for rank in all_rankings
            ],
        )
    )

    fig.update_layout(
        title="Simple Model: Tactical Fit by Coordinate (No Context)",
        xaxis=dict(title="Width", range=[-1, 11], showgrid=False, zeroline=False),
        yaxis=dict(title="Length", range=[-1, 15], showgrid=False, zeroline=False),
        width=700,
        height=900,
        template="plotly_white",
    )

    fig.show()


# Ensure your model is the SimpleTwoTower and loaded with 'simple_soccer_model.pth'
visualize_simple_tactical_fit(
    attacking_433, df_match_real_madrid, model, device, player_stats_cols
)

In [670]:
import plotly.graph_objects as go


def visualize_simple_tactical_fit_with_top3_scores_clean(
    custom_coords, squad_df, model, device, player_stats_cols
):
    """
    Visualize tactical fit with Plotly, showing top 3 players and their scores for every position.
    All positions show top 3 scores as annotations, aligned neatly above the marker.
    """
    model.eval()

    coords_np = np.array(custom_coords, dtype=float)

    # Normalize coordinates as in training
    coords_norm = coords_np.copy()
    coords_norm[:, 0] = (coords_norm[:, 0] - 5.0) / 4.0
    coords_norm[:, 1] = (coords_norm[:, 1] - 7.0) / 4.0

    all_rankings = []
    top3_per_pos = []

    # -------------------------
    # Score players for each position
    # -------------------------
    for i in range(len(coords_norm)):
        target_pos_tensor = torch.tensor([coords_norm[i]], dtype=torch.float32).to(
            device
        )

        with torch.no_grad():
            p_emb = model.position_tower(target_pos_tensor)
            p_norm = p_emb / (torch.linalg.norm(p_emb, dim=1, keepdim=True) + 1e-8)

            scores = []
            for _, player_row in squad_df.iterrows():
                p_stats = torch.tensor(
                    [player_row[player_stats_cols].values], dtype=torch.float32
                ).to(device)

                u_emb = model.player_tower(p_stats)
                u_norm = u_emb / (torch.linalg.norm(u_emb, dim=1, keepdim=True) + 1e-8)

                score = torch.matmul(p_norm, u_norm.T).item()
                scores.append((player_row["player_name"], score))

        # Sort descending
        scores.sort(key=lambda x: x[1], reverse=True)

        # Save top 5 for hover
        top5 = "<br>".join([f"{n}: {s:.4f}" for n, s in scores[:5]])
        all_rankings.append(top5)

        # Save top 3 for annotations
        top3_per_pos.append(scores[:3])

    # -------------------------
    # Plot pitch
    # -------------------------
    fig = go.Figure()

    # Pitch boundaries
    fig.add_shape(type="rect", x0=0, y0=0, x1=10, y1=14, line=dict(color="Black"))

    # Position markers
    fig.add_trace(
        go.Scatter(
            x=coords_np[:, 0],
            y=coords_np[:, 1],
            mode="markers+text",
            marker=dict(
                size=30,
                color="FireBrick",
                line=dict(width=2, color="DarkSlateGrey"),
                opacity=0.8,
            ),
            # text=[f"Pos {i + 1}" for i in range(len(custom_coords))],
            textposition="top center",
            hoverinfo="text",
            hovertext=[f"<b>Top Fits:</b><br>{rank}" for rank in all_rankings],
        )
    )

    # -------------------------
    # Add top 3 annotations for all positions
    # -------------------------
    for i, (x, y) in enumerate(coords_np):
        top3_text = "<br>".join(
            [f"{name}: {score:.4f}" for name, score in top3_per_pos[i]]
        )

        # Use yshift to place text above the marker neatly
        fig.add_annotation(
            x=x,
            y=y,
            text=top3_text,
            showarrow=False,
            font=dict(size=12),
            align="center",
            yshift=50,  # moves the text above the marker
        )

    # -------------------------
    # Layout
    # -------------------------
    fig.update_layout(
        title="Simple Model: Tactical Fit with Top 3 Player Scores",
        xaxis=dict(title="Width", range=[-1, 11], showgrid=False, zeroline=False),
        yaxis=dict(title="Length", range=[-1, 15], showgrid=False, zeroline=False),
        width=1200,
        height=1400,
        template="plotly_white",
    )

    fig.show()


visualize_simple_tactical_fit_with_top3_scores_clean(
    attacking_433, df_match_real_madrid, model, device, player_stats_cols
)

In [671]:
def build_player_position_score_matrix(
    custom_coords,
    squad_df,
    model,
    device,
    player_stats_cols,
):
    """
    Returns a DataFrame of shape (num_players, num_positions)

    Rows: player names
    Columns: pos_1 ... pos_k
    Values: cosine similarity scores between player and position embeddings
    """

    model.eval()

    # ---------------------------
    # 1. Normalize coordinates
    # ---------------------------
    coords_np = np.array(custom_coords, dtype=float)

    coords_norm = coords_np.copy()
    coords_norm[:, 0] = (coords_norm[:, 0] - 5.0) / 4.0
    coords_norm[:, 1] = (coords_norm[:, 1] - 7.0) / 4.0

    pos_tensor = torch.tensor(coords_norm, dtype=torch.float32).to(device)

    # ---------------------------
    # 2. Encode all positions
    # ---------------------------
    with torch.no_grad():
        pos_emb = model.position_tower(pos_tensor)
        pos_norm = pos_emb / (torch.linalg.norm(pos_emb, dim=1, keepdim=True) + 1e-8)

    # ---------------------------
    # 3. Encode all players
    # ---------------------------
    player_stats = squad_df[player_stats_cols].values
    player_tensor = torch.tensor(player_stats, dtype=torch.float32).to(device)

    with torch.no_grad():
        player_emb = model.player_tower(player_tensor)
        player_norm = player_emb / (
            torch.linalg.norm(player_emb, dim=1, keepdim=True) + 1e-8
        )

    # ---------------------------
    # 4. Similarity Matrix
    # ---------------------------
    # (num_players × embedding) @ (embedding × num_positions)
    sim_matrix = torch.matmul(player_norm, pos_norm.T).cpu().numpy()

    # ---------------------------
    # 5. Create DataFrame
    # ---------------------------
    pos_cols = [f"pos_{i + 1}" for i in range(len(custom_coords))]

    df_scores = pd.DataFrame(
        sim_matrix,
        index=squad_df["player_name"].values,
        columns=pos_cols,
    )

    return df_scores

# Hungarian Algorithm

In [ ]:
from scipy.optimize import linear_sum_assignment
import pandas as pd


def find_optimal_lineup(df_scores, position_map=None):
    cost_matrix = -df_scores.values
    row_ind, col_ind = linear_sum_assignment(cost_matrix)

    assignments = []
    total_score = 0.0

    players = df_scores.index.to_list()
    positions = df_scores.columns.to_list()

    for r, c in zip(row_ind, col_ind):
        pos_code = positions[c]
        assignments.append(
            {
                "position_code": pos_code,
                "position": position_map.get(pos_code, pos_code)
                if position_map
                else pos_code,
                "player": players[r],
                "score": df_scores.iloc[r, c],
            }
        )
        total_score += df_scores.iloc[r, c]

    lineup_df = pd.DataFrame(assignments)
    lineup_df["pos_num"] = lineup_df["position_code"].str.extract(r"(\d+)").astype(int)
    lineup_df = (
        lineup_df.sort_values("pos_num").drop(columns="pos_num").reset_index(drop=True)
    )

    return lineup_df, total_score

In [ ]:
# 4-2-3-1 formation coordinates
formation_4231 = [
    (2, 3),
    (8, 3),
    (4, 3),
    (6, 3),  # Back 4
    (4, 6),
    (6, 6),  # Midfield 2
    (2, 9),
    (5, 9),
    (8, 9),  # Attacking Mid 3
    (5, 12),  # Striker
]

# 4-3-3 formation coordinates
formation_433 = [
    (2, 3),  # LB
    (4, 3),  # LCB
    (6, 3),  # RCB
    (8, 3),  # RB
    (3, 7),  # CDM
    (5, 7),  # LCM
    (7, 7),  # RCM
    (2, 11),  # LW
    (8, 11),  # RW
    (5, 11),  # ST
]

In [ ]:
# Exact mapping based on the ORDER of coordinates in formation_4231
position_map_4231 = {
    "pos_1": "LB",  # (2, 3)
    "pos_2": "RB",  # (8, 3)
    "pos_3": "LCB",  # (4, 3)
    "pos_4": "RCB",  # (6, 3)
    "pos_5": "LDM",  # (4, 6)
    "pos_6": "RDM",  # (6, 6)
    "pos_7": "LW",  # (2, 9)
    "pos_8": "CAM",  # (5, 9)
    "pos_9": "RW",  # (8, 9)
    "pos_10": "ST",  # (5, 12)
}

# Exact mapping based on the ORDER of coordinates in formation_433
position_map_433 = {
    "pos_1": "LB",  # (2, 3)
    "pos_2": "LCB",  # (4, 3)
    "pos_3": "RCB",  # (6, 3)
    "pos_4": "RB",  # (8, 3)
    "pos_5": "LCM",  # (3, 7)
    "pos_6": "CDM",  # (5, 7)
    "pos_7": "RCM",  # (7, 7)
    "pos_8": "LW",  # (2, 11)
    "pos_9": "RW",  # (8, 11)
    "pos_10": "ST",  # (5, 11)
}

# Visualization

In [ ]:
import plotly.graph_objects as go
import numpy as np


def visualize_optimal_lineup(
    custom_coords,
    df_scores,
    lineup_df,
    total_score,
    formation_name="Formation",
    pitch_width=10,
    pitch_length=14,
):
    """
    Visualize the optimal lineup on a soccer pitch.

    Parameters
    ----------
    custom_coords : list of tuples
        Formation coordinates in the SAME ORDER used to build df_scores
    df_scores : pd.DataFrame
        Player-position similarity matrix
    lineup_df : pd.DataFrame
        Output from find_optimal_lineup()
    total_score : float
        Total score of the selected lineup
    formation_name : str
        Title label for the chart
    pitch_width : int/float
        Width of pitch for plotting
    pitch_length : int/float
        Length of pitch for plotting
    """

    coords_np = np.array(custom_coords, dtype=float)

    # -----------------------------------
    # Build top-3 hover text for each position
    # -----------------------------------
    hover_texts = []
    for j, pos_col in enumerate(df_scores.columns):
        top3 = df_scores[pos_col].sort_values(ascending=False).head(3)
        top3_text = "<br>".join(
            [f"{player}: {score:.4f}" for player, score in top3.items()]
        )
        hover_texts.append(f"<b>{pos_col} top 3 fits:</b><br>{top3_text}")

    # -----------------------------------
    # Build display labels for selected players
    # -----------------------------------
    marker_text = []
    for i in range(len(lineup_df)):
        role = lineup_df.loc[i, "position"]
        player = lineup_df.loc[i, "player"]
        score = lineup_df.loc[i, "score"]

        marker_text.append(f"{role}<br>{player}<br>{score:.3f}")

    # -----------------------------------
    # Plot pitch
    # -----------------------------------
    fig = go.Figure()

    # Outer pitch boundary
    fig.add_shape(
        type="rect",
        x0=0,
        y0=0,
        x1=pitch_width,
        y1=pitch_length,
        line=dict(color="black", width=2),
    )

    # Halfway line
    fig.add_shape(
        type="line",
        x0=0,
        y0=pitch_length / 2,
        x1=pitch_width,
        y1=pitch_length / 2,
        line=dict(color="black", width=1),
    )

    # Center circle
    fig.add_shape(
        type="circle",
        x0=pitch_width / 2 - 1,
        y0=pitch_length / 2 - 1,
        x1=pitch_width / 2 + 1,
        y1=pitch_length / 2 + 1,
        line=dict(color="black", width=1),
    )

    # -----------------------------------
    # Add chosen lineup markers
    # -----------------------------------
    fig.add_trace(
        go.Scatter(
            x=coords_np[:, 0],
            y=coords_np[:, 1],
            mode="markers+text",
            marker=dict(
                size=34,
                color="royalblue",
                line=dict(width=2, color="darkblue"),
                opacity=0.9,
            ),
            text=marker_text,
            textposition="top center",
            hoverinfo="text",
            hovertext=hover_texts,
        )
    )

    # -----------------------------------
    # Layout
    # -----------------------------------
    fig.update_layout(
        title=f"Optimal Lineup ({formation_name})<br>Total Similarity Score = {total_score:.3f}",
        xaxis=dict(
            title="Pitch Width",
            range=[-0.5, pitch_width + 0.5],
            showgrid=False,
            zeroline=False,
            visible=False,
        ),
        yaxis=dict(
            title="Pitch Length",
            range=[-0.5, pitch_length + 0.5],
            showgrid=False,
            zeroline=False,
            visible=False,
            scaleanchor="x",
            scaleratio=1,
        ),
        width=900,
        height=1100,
        template="plotly_white",
        showlegend=False,
    )

    fig.show()

# Test

### Bayern FC: 4-2-3-1 Formation

In [676]:
df_scores_fc_bayern_4231 = build_player_position_score_matrix(
    modern_4231,
    df_match_fc_bayern_munich,
    model,
    device,
    player_stats_cols,
)

print(df_scores_fc_bayern_4231)
df_scores_fc_bayern_4231.to_csv("fc_bayern_4231.csv")

                       pos_1     pos_2     pos_3     pos_4     pos_5  \
Arjen Robben        0.124780  0.305692  0.288063  0.230453  0.428426   
Philipp Lahm        0.385011  0.583792  0.470504  0.462184  0.608280   
Arturo Vidal        0.213661  0.431400  0.405404  0.331081  0.588404   
Robert Lewandowski  0.032535  0.264386  0.255284  0.217544  0.397008   
Thomas Mueller      0.075614  0.319756  0.308479  0.264033  0.436594   
David Alaba         0.304156  0.532718  0.487869  0.418376  0.595715   
Douglas Costa       0.206006  0.399876  0.345440  0.270082  0.499577   
Thiago Alcantara    0.201668  0.383750  0.384468  0.314391  0.545585   
Juan Bernat         0.330524  0.485820  0.464764  0.400702  0.579413   
Joshua Kimmich      0.261357  0.391720  0.448465  0.368533  0.612443   

                       pos_6     pos_7     pos_8     pos_9    pos_10  
Arjen Robben        0.516163  0.321132  0.510380  0.590700  0.421849  
Philipp Lahm        0.708822  0.135408  0.295184  0.346459  0.035

In [ ]:
optimal_lineup, total_score = find_optimal_lineup(
    df_scores_fc_bayern_4231, position_map=position_map_4231
)

print("Optimal Lineup:")
print(optimal_lineup)

print("\nTotal Similarity Score:", total_score)

Optimal Lineup:
  position_code position              player     score
0         pos_1       LB         Juan Bernat  0.330524
1         pos_2       RB        Philipp Lahm  0.583792
2         pos_3      LCB    Thiago Alcantara  0.384468
3         pos_4      RCB         David Alaba  0.418376
4         pos_5      LDM      Joshua Kimmich  0.612443
5         pos_6      RDM        Arturo Vidal  0.685518
6         pos_7       LW        Arjen Robben  0.321132
7         pos_8      CAM      Thomas Mueller  0.487550
8         pos_9       RW       Douglas Costa  0.603147
9        pos_10       ST  Robert Lewandowski  0.458250

Total Similarity Score: 4.885200470685959


In [ ]:
visualize_optimal_lineup(
    custom_coords=formation_4231,
    df_scores=df_scores_fc_bayern_4231,
    lineup_df=optimal_lineup,
    total_score=total_score,
    formation_name="4-2-3-1",
)

### Bayern FC: 4-3-3 Formation

In [679]:
df_scores_fc_bayern_433 = build_player_position_score_matrix(
    attacking_433,
    df_match_fc_bayern_munich,
    model,
    device,
    player_stats_cols,
)

print(df_scores_fc_bayern_433)
df_scores_fc_bayern_433.to_csv("fc_bayern_433.csv")

                       pos_1     pos_2     pos_3     pos_4     pos_5  \
Arjen Robben        0.124780  0.288063  0.230453  0.305692  0.353282   
Philipp Lahm        0.385011  0.470504  0.462184  0.583792  0.455702   
Arturo Vidal        0.213661  0.405404  0.331081  0.431400  0.335626   
Robert Lewandowski  0.032535  0.255284  0.217544  0.264386  0.276952   
Thomas Mueller      0.075614  0.308479  0.264033  0.319756  0.300880   
David Alaba         0.304156  0.487869  0.418376  0.532718  0.353640   
Douglas Costa       0.206006  0.345440  0.270082  0.399876  0.353406   
Thiago Alcantara    0.201668  0.384468  0.314391  0.383750  0.417231   
Juan Bernat         0.330524  0.464764  0.400702  0.485820  0.394783   
Joshua Kimmich      0.261357  0.448465  0.368533  0.391720  0.420206   

                       pos_6     pos_7     pos_8     pos_9    pos_10  
Arjen Robben        0.452022  0.490284  0.271375  0.539200  0.463130  
Philipp Lahm        0.461615  0.647655  0.085397  0.270036  0.061

In [ ]:
optimal_lineup, total_score = find_optimal_lineup(
    df_scores_fc_bayern_433, position_map=position_map_433
)

print("Optimal Lineup:")
print(optimal_lineup)

print("\nTotal Similarity Score:", total_score)

Optimal Lineup:
  position_code position              player     score
0         pos_1       LB         Juan Bernat  0.330524
1         pos_2      LCB      Joshua Kimmich  0.448465
2         pos_3      RCB         David Alaba  0.418376
3         pos_4       RB        Philipp Lahm  0.583792
4         pos_5      LCM    Thiago Alcantara  0.417231
5         pos_6      CDM      Thomas Mueller  0.470682
6         pos_7      RCM        Arturo Vidal  0.544895
7         pos_8       LW        Arjen Robben  0.271375
8         pos_9       RW       Douglas Costa  0.551244
9        pos_10       ST  Robert Lewandowski  0.483892

Total Similarity Score: 4.520476311445236


In [ ]:
visualize_optimal_lineup(
    custom_coords=formation_433,
    df_scores=df_scores_fc_bayern_433,
    lineup_df=optimal_lineup,
    total_score=total_score,
    formation_name="4-3-3",
)

### Real Madrid: 4-2-3-1 Formation

In [682]:
df_scores_real_madrid_4231 = build_player_position_score_matrix(
    modern_4231,
    df_match_real_madrid,
    model,
    device,
    player_stats_cols,
)

print(df_scores_real_madrid_4231)
df_scores_real_madrid_4231.to_csv("real_madrid_4231.csv")

                      pos_1     pos_2     pos_3     pos_4     pos_5     pos_6  \
Pepe               0.309173  0.497334  0.553500  0.468837  0.572244  0.653223   
Karim Benzema      0.062199  0.258589  0.199926  0.156697  0.348594  0.431160   
Marcelo            0.313405  0.480282  0.456023  0.374091  0.616794  0.709430   
Cristiano Ronaldo  0.059026  0.328616  0.276202  0.239268  0.367524  0.475762   
Sergio Ramos       0.234908  0.494788  0.479122  0.420795  0.586117  0.679492   
Luka Modric        0.257164  0.454603  0.428307  0.380971  0.592991  0.693509   
Gareth Bale        0.159772  0.375723  0.366049  0.301405  0.476433  0.583736   
Toni Kroos         0.118909  0.379113  0.387194  0.380050  0.547266  0.683039   
Casemiro           0.189296  0.423436  0.452021  0.367550  0.612442  0.699684   
Daniel Carvajal    0.343214  0.492986  0.481686  0.436529  0.559246  0.666529   

                      pos_7     pos_8     pos_9    pos_10  
Pepe               0.048624  0.261799  0.286160 

In [ ]:
optimal_lineup, total_score = find_optimal_lineup(
    df_scores_real_madrid_4231, position_map=position_map_4231
)

print("Optimal Lineup:")
print(optimal_lineup)

print("\nTotal Similarity Score:", total_score)

Optimal Lineup:
  position_code position             player     score
0         pos_1       LB            Marcelo  0.313405
1         pos_2       RB       Sergio Ramos  0.494788
2         pos_3      LCB               Pepe  0.553500
3         pos_4      RCB    Daniel Carvajal  0.436529
4         pos_5      LDM           Casemiro  0.612442
5         pos_6      RDM        Luka Modric  0.693509
6         pos_7       LW        Gareth Bale  0.226583
7         pos_8      CAM         Toni Kroos  0.423233
8         pos_9       RW  Cristiano Ronaldo  0.636685
9        pos_10       ST      Karim Benzema  0.536058

Total Similarity Score: 4.926732197403908


In [ ]:
visualize_optimal_lineup(
    custom_coords=formation_4231,
    df_scores=df_scores_real_madrid_4231,
    lineup_df=optimal_lineup,
    total_score=total_score,
    formation_name="4-2-3-1",
)

### Real Madrid: 4-3-3 Formation

In [685]:
df_scores_real_madrid_433 = build_player_position_score_matrix(
    attacking_433,
    df_match_real_madrid,
    model,
    device,
    player_stats_cols,
)

print(df_scores_real_madrid_433)
df_scores_real_madrid_433.to_csv("real_madrid_433.csv")

                      pos_1     pos_2     pos_3     pos_4     pos_5     pos_6  \
Pepe               0.309173  0.553500  0.468837  0.497334  0.380722  0.507049   
Karim Benzema      0.062199  0.199926  0.156697  0.258589  0.217087  0.412759   
Marcelo            0.313405  0.456023  0.374091  0.480282  0.377155  0.477566   
Cristiano Ronaldo  0.059026  0.276202  0.239268  0.328616  0.226815  0.449516   
Sergio Ramos       0.234908  0.479122  0.420795  0.494788  0.297312  0.532677   
Luka Modric        0.257164  0.428307  0.380971  0.454603  0.421389  0.507041   
Gareth Bale        0.159772  0.366049  0.301405  0.375723  0.269882  0.448301   
Toni Kroos         0.118909  0.387194  0.380050  0.379113  0.327015  0.510632   
Casemiro           0.189296  0.452021  0.367550  0.423436  0.366498  0.518278   
Daniel Carvajal    0.343214  0.481686  0.436529  0.492986  0.398993  0.454239   

                      pos_7     pos_8     pos_9    pos_10  
Pepe               0.563117  0.023964  0.230077 

In [ ]:
optimal_lineup, total_score = find_optimal_lineup(
    df_scores_real_madrid_433, position_map=position_map_433
)

print("Optimal Lineup:")
print(optimal_lineup)

print("\nTotal Similarity Score:", total_score)

Optimal Lineup:
  position_code position             player     score
0         pos_1       LB    Daniel Carvajal  0.343214
1         pos_2      LCB               Pepe  0.553500
2         pos_3      RCB         Toni Kroos  0.380050
3         pos_4       RB       Sergio Ramos  0.494788
4         pos_5      LCM        Luka Modric  0.421389
5         pos_6      CDM           Casemiro  0.518278
6         pos_7      RCM            Marcelo  0.586845
7         pos_8       LW        Gareth Bale  0.213262
8         pos_9       RW  Cristiano Ronaldo  0.589816
9        pos_10       ST      Karim Benzema  0.558977

Total Similarity Score: 4.660118505358696


In [ ]:
visualize_optimal_lineup(
    custom_coords=formation_433,
    df_scores=df_scores_real_madrid_433,
    lineup_df=optimal_lineup,
    total_score=total_score,
    formation_name="4-3-3",
)

### Fantasy Team: 4-2-3-1 Formation

(pick b/w players from Madrid & Bayern FC)

In [ ]:
df_fantasy_squad = pd.concat(
    [df_match_real_madrid, df_match_fc_bayern_munich], ignore_index=True
)

df_fantasy_squad = df_fantasy_squad.drop_duplicates(subset="player_name")

In [689]:
df_scores_fantasy_4231 = build_player_position_score_matrix(
    modern_4231,
    df_fantasy_squad,
    model,
    device,
    player_stats_cols,
)

print(df_scores_fantasy_4231)

                       pos_1     pos_2     pos_3     pos_4     pos_5  \
Pepe                0.309173  0.497334  0.553500  0.468837  0.572244   
Karim Benzema       0.062199  0.258589  0.199926  0.156697  0.348594   
Marcelo             0.313405  0.480282  0.456023  0.374091  0.616794   
Cristiano Ronaldo   0.059026  0.328616  0.276202  0.239268  0.367524   
Sergio Ramos        0.234908  0.494788  0.479122  0.420795  0.586117   
Luka Modric         0.257164  0.454603  0.428307  0.380971  0.592991   
Gareth Bale         0.159772  0.375723  0.366049  0.301405  0.476433   
Toni Kroos          0.118909  0.379113  0.387194  0.380050  0.547266   
Casemiro            0.189296  0.423436  0.452021  0.367550  0.612442   
Daniel Carvajal     0.343214  0.492986  0.481686  0.436529  0.559246   
Arjen Robben        0.124780  0.305692  0.288063  0.230453  0.428426   
Philipp Lahm        0.385011  0.583792  0.470504  0.462184  0.608280   
Arturo Vidal        0.213661  0.431400  0.405404  0.331081  0.58

In [ ]:
optimal_lineup, total_score = find_optimal_lineup(
    df_scores_fantasy_4231, position_map=position_map_433
)

print("Optimal Lineup:")
print(optimal_lineup)

print("\nTotal Similarity Score:", total_score)

Optimal Lineup:
  position_code position             player     score
0         pos_1       LB       Philipp Lahm  0.385011
1         pos_2      LCB        David Alaba  0.532718
2         pos_3      RCB               Pepe  0.553500
3         pos_4       RB    Daniel Carvajal  0.436529
4         pos_5      LCM     Joshua Kimmich  0.612443
5         pos_6      CDM            Marcelo  0.709430
6         pos_7      RCM      Douglas Costa  0.313358
7         pos_8       LW       Arjen Robben  0.510380
8         pos_9       RW  Cristiano Ronaldo  0.636685
9        pos_10       ST      Karim Benzema  0.536058

Total Similarity Score: 5.226111471652985


In [ ]:
visualize_optimal_lineup(
    custom_coords=formation_4231,
    df_scores=df_scores_fantasy_4231,
    lineup_df=optimal_lineup,
    total_score=total_score,
    formation_name="4-2-3-1",
)

### Fantasy Team: 4-3-3 Formation

In [692]:
df_scores_fantasy_433 = build_player_position_score_matrix(
    attacking_433,
    df_fantasy_squad,
    model,
    device,
    player_stats_cols,
)

print(df_scores_fantasy_433)

                       pos_1     pos_2     pos_3     pos_4     pos_5  \
Pepe                0.309173  0.553500  0.468837  0.497334  0.380722   
Karim Benzema       0.062199  0.199926  0.156697  0.258589  0.217087   
Marcelo             0.313405  0.456023  0.374091  0.480282  0.377155   
Cristiano Ronaldo   0.059026  0.276202  0.239268  0.328616  0.226815   
Sergio Ramos        0.234908  0.479122  0.420795  0.494788  0.297312   
Luka Modric         0.257164  0.428307  0.380971  0.454603  0.421389   
Gareth Bale         0.159772  0.366049  0.301405  0.375723  0.269882   
Toni Kroos          0.118909  0.387194  0.380050  0.379113  0.327015   
Casemiro            0.189296  0.452021  0.367550  0.423436  0.366498   
Daniel Carvajal     0.343214  0.481686  0.436529  0.492986  0.398993   
Arjen Robben        0.124780  0.288063  0.230453  0.305692  0.353282   
Philipp Lahm        0.385011  0.470504  0.462184  0.583792  0.455702   
Arturo Vidal        0.213661  0.405404  0.331081  0.431400  0.33

In [ ]:
optimal_lineup, total_score = find_optimal_lineup(
    df_scores_fantasy_433, position_map=position_map_433
)

print("Optimal Lineup:")
print(optimal_lineup)

print("\nTotal Similarity Score:", total_score)

Optimal Lineup:
  position_code position             player     score
0         pos_1       LB       Philipp Lahm  0.385011
1         pos_2      LCB               Pepe  0.553500
2         pos_3      RCB    Daniel Carvajal  0.436529
3         pos_4       RB        David Alaba  0.532718
4         pos_5      LCM     Joshua Kimmich  0.420206
5         pos_6      CDM       Sergio Ramos  0.532677
6         pos_7      RCM        Luka Modric  0.611274
7         pos_8       LW      Douglas Costa  0.271408
8         pos_9       RW  Cristiano Ronaldo  0.589816
9        pos_10       ST      Karim Benzema  0.558977

Total Similarity Score: 4.892117530107498


In [ ]:
visualize_optimal_lineup(
    custom_coords=formation_433,
    df_scores=df_scores_fantasy_433,
    lineup_df=optimal_lineup,
    total_score=total_score,
    formation_name="4-3-3",
)